# Crear les dades inicials amb els paràmetres inicials

In [11]:
import pandas as pd
import os
from pathlib import Path
from tqdm import tqdm
from datetime import datetime

In [12]:
def simulate_tour(tour_df:pd.DataFrame, players_df:pd.DataFrame, k, ksi, s, initial_elo, min_games, year_to_simulate): 
    assert tour_df.isna().sum().sum() == 0, f'nan values in tour_df\n{tour_df.isna().sum()}'

    tour_df = tour_df.sort_values(by='tourney_date', ascending=True)

    if year_to_simulate != 'Tots': 
        tour_df = tour_df[tour_df['match_year']==year_to_simulate]

    players_dic = {player_id: {
            'player_id': player_id, 
            'elo_rating': initial_elo, 
            'elo_clay_rating': initial_elo, 
            'elo_hard_rating': initial_elo, 
            'elo_grass_rating': initial_elo, 
            'elo_carpet_rating': initial_elo, 
            'elo_unknown_rating': initial_elo,
            'n_games': 0,
            'last_game': None,
            'n_wins': 0,
            'n_losses': 0
        } for player_id in players_df['player_id']}

    elo_history_list = []

    year = ''
    for _, m in tour_df.iterrows():
        
        if year != m['match_year']:
            # placeholder.write(f'    Simulating year {m['match_year']}...')
            year = m['match_year']

        # Update elo ratings
        match s:
            case 'delta':
                Sw = 1
                Sl = 0

            case 'thirds': 
                if m['best_of'] != m['num_sets'] or m['best_of'] == 1:
                    Sw = 1
                    Sl = 0
                else: 
                    Sw = 2/3
                    Sl = 1/3

        
        # Algorisme per calcular elo-ratings
        winner_id = m['winner_id']
        loser_id = m['loser_id']
        elo_surface = f'elo_{m['surface'].lower()}_rating'
        match_date = m['tourney_date']
        surface = m['surface']

        old_wr =  players_dic[winner_id]['elo_rating']
        old_lr = players_dic[loser_id]['elo_rating']
        # Surface
        old_slr = players_dic[winner_id][elo_surface]
        old_swr = players_dic[loser_id][elo_surface]

        mu_w = 1 / (1 + pow(10, -(old_wr - old_lr)/ksi))
        mu_l = 1 / (1 + pow(10, -(old_lr - old_wr)/ksi))
        # Surface
        mu_sw = 1 / (1 + pow(10, -(old_swr - old_slr)/ksi))
        mu_sl = 1 / (1 + pow(10, -(old_slr - old_swr)/ksi))

        # Actualitzar els valors dels elo-ratings dels jugadors. 
        winner_new_elo = old_wr + k*(Sw - mu_w)
        loser_new_elo = old_lr + k*(Sl - mu_l)
        players_dic[winner_id]['elo_rating'] = winner_new_elo
        players_dic[loser_id]['elo_rating'] = loser_new_elo
        # Surface
        players_dic[winner_id][elo_surface] = old_swr + k*(Sw - mu_sw)
        players_dic[loser_id][elo_surface] = old_slr + k*(Sl - mu_sl)

        players_dic[loser_id]['n_games'] += 1
        players_dic[loser_id]['last_game'] = match_date
        players_dic[winner_id]['n_games'] += 1
        players_dic[winner_id]['last_game'] = match_date
        players_dic[winner_id]['n_wins'] += 1
        players_dic[loser_id]['n_losses'] += 1
        assert players_dic[winner_id]['n_games'] == players_dic[winner_id]['n_wins'] + players_dic[winner_id]['n_losses'],\
            f"Error: n_games != n_wins + n_losses -> {players_dic[winner_id]['n_games']} != {players_dic[winner_id]['n_wins']} + {players_dic[winner_id]['n_losses']}"

        # Històric
        winner_history = {
            'player_id': winner_id,
            'date': match_date,
            'elo_rating': winner_new_elo
        }
        loser_history = {
            'player_id': loser_id,
            'date': match_date,
            'elo_rating': loser_new_elo
        }

        elo_history_list.append(winner_history)
        elo_history_list.append(loser_history)


    players_simulated: pd.DataFrame = pd.DataFrame.from_dict(players_dic, orient='index')

    ranking: pd.DataFrame = players_df.merge(players_simulated, how='left', on='player_id')\
        .sort_values(by='elo_rating', ascending=False)\
        .reset_index(drop=True)\
        .drop(columns=['elo_unknown_rating'])\
        .round(0)\
        .rename(columns={
            'elo_rating': 'Elo Rating', 
            'elo_clay_rating': 'Clay Elo Rating', 
            'elo_hard_rating': 'Hard Elo Rating', 
            'elo_grass_rating': 'Grass Elo Rating', 
            'elo_carpet_rating': 'Carpet Elo Rating'
        })


    ranking_filtered = ranking[(ranking['n_games']>min_games)]# & (ranking['last_game'] > '2023-01-01')].reset_index(drop=True)

    ranking_filtered['rank'] = ranking_filtered.index + 1

    elo_history_df = pd.DataFrame(elo_history_list)\
                        .astype({'date': 'datetime64[ns]'})

    return ranking_filtered, elo_history_df


In [13]:
k = 24
ksi = 400
s = 'delta'
initial_elo = 1500
min_games = 30
year_to_simulate = 'Tots'

In [14]:
atp_matches_df = pd.read_csv('../web_data/clean_atp_matches.csv')
atp_players_df = pd.read_csv('../web_data/clean_atp_players.csv')
wta_matches_df = pd.read_csv('../web_data/clean_wta_matches.csv')
wta_players_df = pd.read_csv('../web_data/clean_wta_players.csv')
assert atp_matches_df.isna().sum().sum() == 0, f'nan values in matches\n{atp_matches_df.isna().sum()}'
assert atp_players_df.isna().sum().sum() == 0, f'nan values in players\n{atp_players_df.isna().sum()}'
assert atp_matches_df.isna().sum().sum() == 0, f'nan values in matches\n{wta_matches_df.isna().sum()}'
assert atp_players_df.isna().sum().sum() == 0, f'nan values in players\n{wta_players_df.isna().sum()}'

In [15]:
atp_ranking, atp_elo_history = simulate_tour(atp_matches_df, atp_players_df, k, ksi, s, initial_elo, min_games, year_to_simulate)
atp_ranking.index += 1
wta_ranking, wta_elo_history = simulate_tour(wta_matches_df, wta_players_df, k, ksi, s, initial_elo, min_games, year_to_simulate)
wta_ranking.index += 1

/var/folders/s3/s4gmfm1j23xg8_5fnbsqlflm0000gp/T/ipykernel_52674/1637771482.py:118: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  ranking_filtered['rank'] = ranking_filtered.index + 1
/var/folders/s3/s4gmfm1j23xg8_5fnbsqlflm0000gp/T/ipykernel_52674/1637771482.py:118: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  ranking_filtered['rank'] = ranking_filtered.index + 1


In [16]:
atp_elo_history.to_csv('../web_data/atp_initial_elo_history.csv')
atp_ranking.to_csv('../web_data/atp_initial_ranking.csv')
wta_elo_history.to_csv('../web_data/wta_initial_elo_history.csv')
wta_ranking.to_csv('../web_data/wta_initial_ranking.csv')